# Smart Agriculture: Soil, Fertilizer and Crop Data Analysis with Machine Learning

**Companion code for the paper:**
Mishra, B. K., Singh, B., & Panwar, R. (2025). *Smart agriculture using deep learning and FDM for the relation of soil health, fertilizer and ground water.* International Journal of System Assurance Engineering and Management. https://doi.org/10.1007/s13198-025-03055-1

---

### Scope: what this notebook covers
It reproduces the data analysis and model comparison reported in Section 3 (Result and discussion) of the paper:

| Paper item | Where in this notebook |
|---|---|
| Table 1: data from various sources (columns, non-null counts, data types) | Steps 3 and 4 |
| Table 2: values of different soil parameters | Step 6 |
| Fig. 3: data visualization of soil parameters | Step 8b |
| Fig. 4: accuracy on the whole data for the trained models | Step 12 |
| Fig. 5: accuracy on the test data for the trained models | Step 12 |

Six classical machine-learning classifiers are compared on 2,200 samples of 22 crops: Logistic Regression, K-Nearest Neighbors, Random Forest, XGBoost, Decision Tree and Gaussian Naive Bayes.

**Not included here:** the conceptual framework (Figs. 1 and 2) and the symbolic Eqs. (1) to (10) of the paper are not implemented as code in this notebook, and the notebook contains no neural network and no finite-difference solver.

### Steps (each step has a short description, the code, and citations where a method is used)
1. Import libraries
2. Configuration
3. Load the dataset
4. Inspect the data
5. Missing-value treatment
6. Descriptive statistics
7. Outlier check (IQR rule)
8. Exploratory analysis (class balance, distributions, feature vs crop, correlation)
9. Label encoding and train/test split
10. Model evaluation helper
11. The six classifiers (one step each)
12. Model comparison
13. Robustness check with cross-validation (additional, not part of Figs. 4 and 5)
14. Limitations and references

### How to run
Place the dataset in `data/` (see `data/README.md`), install `requirements.txt`, then choose **Kernel > Restart & Run All**. Figures are saved to `results/`.

## Step 1. Import libraries

**What:** loads the Python libraries used throughout the notebook.
**Why:** `pandas` and `numpy` handle the data, `matplotlib` and `seaborn` draw the figures, and `scikit-learn` provides preprocessing, models and metrics. `xgboost` provides the boosting classifier.

**Citations:** pandas (McKinney, 2010); NumPy (Harris et al., 2020); Matplotlib (Hunter, 2007); seaborn (Waskom, 2021); scikit-learn (Pedregosa et al., 2011); XGBoost (Chen & Guestrin, 2016). Full references are listed at the end.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")

## Step 2. Configuration

**What:** sets file locations, random seeds and a few switches in one place.
**Why:** no hard-coded personal paths, so the notebook runs on any computer, and the fixed seeds make the results repeatable.

* `SEED_SPLIT = 0` and `TEST_SIZE = 0.2` give the 80/20 train/test split used in the paper.
* `RUN_PAIRPLOT` is `False` by default because the full pair plot of 13 features and 22 classes is slow and produces a very large image. Set it to `True` to reproduce it.
* `SAVE_EDA_LOOPS` controls whether the many per-feature figures are also saved to `results/`.

In [ ]:
SEED_SPLIT = 0          # random_state of the train/test split
TEST_SIZE = 0.2         # 20 % of the samples are held out for testing
K_FINAL = 3             # number of neighbours used for the final KNN model (see Step 11b)
RUN_PAIRPLOT = False    # True = also draw the (slow) full pair plot
SAVE_FIGURES = True     # save key figures to results/
SAVE_EDA_LOOPS = False  # also save every per-feature figure (many files)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

# The full dataset is used if present, otherwise the small sample (see data/README.md)
candidates = [ROOT / "data" / "Crop_recommendation.csv", ROOT / "data" / "sample_crop_data.csv"]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Put Crop_recommendation.csv (or sample_crop_data.csv) in the data/ folder. See data/README.md.")
if DATA_PATH.name.startswith("sample"):
    print("WARNING: running on the small sample. Numbers will differ from the paper.")

def save_fig(name, loop=False):
    # Save the current matplotlib figure into results/.
    if SAVE_FIGURES and (SAVE_EDA_LOOPS or not loop):
        plt.savefig(RESULTS / f"{name}.png", dpi=120, bbox_inches="tight")

print("Data file:", DATA_PATH)

## Step 3. Load the dataset

**What:** reads the CSV file into a pandas table (`df`).

**The columns:**

| Column | Meaning |
|---|---|
| N, P, K | Nitrogen, phosphorus and potassium content of the soil |
| S, Cu, Fe, Mn, Zn, B | Sulphur and micronutrients (copper, iron, manganese, zinc, boron) |
| ph | Soil pH |
| temperature, humidity, rainfall | Weather variables |
| label | The crop (22 categories) |

**Data source:** see `data/README.md`. The seven columns N, P, K, temperature, humidity, ph and rainfall follow the public *Crop Recommendation Dataset* (Ingle, n.d.).

In [ ]:
df = pd.read_csv(DATA_PATH)
feature_cols = [c for c in df.columns if c != "label"]
print(f"Loaded {DATA_PATH.name}: {df.shape[0]} rows x {df.shape[1]} columns")
print("Features:", feature_cols)

## Step 4. Inspect the data

**What:** shows the first and last rows, the data types and the number of missing values per column.
**Why:** to confirm the file loaded correctly and to find columns that need cleaning. The column names, non-null counts and data types shown by `df.info()` correspond to **Table 1** of the paper.

In [ ]:
display(df.head())
display(df.tail())
df.info()

In [ ]:
print("Missing values per column:")
print(df.isna().sum())

## Step 5. Missing-value treatment

**What:** replaces each missing value with the **mean** of its column (mean imputation).
**Why:** the classifiers cannot work with empty cells. A copy of the raw table (`df_raw`) is kept for the cross-validation check in Step 13.

**Note:** following the paper, the means are computed on the whole table before the train/test split. A stricter approach puts the imputer inside the model pipeline so that the means come from the training data only. Step 13 does exactly that.

In [ ]:
df_raw = df.copy()                                    # raw copy, still contains the gaps
mean_values = df[feature_cols].mean()
df[feature_cols] = df[feature_cols].fillna(mean_values)

print("Missing values after imputation:", int(df.isna().sum().sum()))

## Step 6. Descriptive statistics

**What:** count, mean, standard deviation, minimum, quartiles and maximum of every feature.
**Why:** gives the typical range of each soil and weather variable and reveals unusual values early. This is **Table 2** of the paper.

In [ ]:
df[feature_cols].describe().T

## Step 7. Outlier check (IQR rule)

**What:** for potassium (K) in each crop, computes the lower and upper limits
`Q1 - 1.5 x IQR` and `Q3 + 1.5 x IQR` and tests whether the minimum or maximum value of that crop falls outside them.
**Why:** to see whether extreme measurements could distort the models. This step is a **diagnostic only**: no rows are removed.

**Citation:** the 1.5 x IQR rule is the standard boxplot rule from Tukey (1977).

In [ ]:
rows = []
for crop in df["label"].unique():
    x = df.loc[df["label"] == crop, "K"]
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rows.append({"crop": crop, "lower_limit": lower, "upper_limit": upper,
                 "min": x.min(), "max": x.max(),
                 "min_is_outlier": x.min() < lower, "max_is_outlier": x.max() > upper})

outlier_summary = pd.DataFrame(rows)
outlier_summary

## Step 8. Exploratory data analysis (EDA)

EDA means looking at the data with simple charts before modelling (Tukey, 1977).

### 8a. Number of samples per crop
Shows whether the crops are equally represented. Unequal class sizes would make accuracy harder to interpret.

In [ ]:
plt.figure(figsize=(12, 5))
df["label"].value_counts().plot(kind="bar", color="#4C8C4A")
plt.xlabel("Crop (category)")
plt.ylabel("Number of samples")
plt.title("Number of samples per crop")
save_fig("01_class_distribution")
plt.show()

### 8b. Distribution of every feature
Histograms show the shape and spread of each variable (skewness, long tails, clusters). **Fig. 3** of the paper shows these histograms.

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(15, 17))
for ax, col in zip(axes.ravel(), feature_cols):
    sns.histplot(df[col], ax=ax)
    ax.set_title(col)
for ax in axes.ravel()[len(feature_cols):]:
    ax.set_visible(False)
plt.tight_layout()
save_fig("02_feature_histograms")
plt.show()

### 8c. Average value of each feature for each crop
Bar charts (mean with confidence interval) show which soil and weather conditions are typical for each crop. This is the basis for crop-specific nutrient management.

In [ ]:
for col in feature_cols:
    plt.figure(figsize=(14, 5))
    sns.barplot(x="label", y=col, data=df)
    plt.xticks(rotation=90)
    plt.title(f"{col} vs crop type")
    save_fig(f"eda_bar_{col}", loop=True)
    plt.show()

### 8d. Distribution of each feature with a smooth density curve
Histogram plus kernel density estimate for every feature.

In [ ]:
for col in feature_cols:
    sns.displot(x=col, data=df, bins=20, kde=True, color="#ffb03b",
                edgecolor="black", height=4, aspect=1.8)
    plt.title(f"Distribution of {col}")
    save_fig(f"eda_dist_{col}", loop=True)
    plt.show()

### 8e. Rainfall versus temperature, coloured by crop
Shows how well two weather variables alone separate the crops.

In [ ]:
sns.relplot(x="rainfall", y="temperature", data=df, kind="scatter",
            hue="label", height=6, aspect=1.4)
save_fig("03_rainfall_vs_temperature")
plt.show()

### 8f. Pair plot (optional)
All pairs of features in one grid. It is slow with 13 features and 22 crops, so it only runs when `RUN_PAIRPLOT = True` in Step 2.

In [ ]:
if RUN_PAIRPLOT:
    sns.pairplot(data=df, hue="label", corner=True)
    save_fig("04_pairplot")
    plt.show()
else:
    print("Pair plot skipped (set RUN_PAIRPLOT = True in Step 2 to draw it).")

### 8g. Box plot of every feature for each crop
Box plots show the median, spread and extreme values of each feature within each crop.

In [ ]:
for col in feature_cols:
    sns.catplot(x="label", y=col, data=df, kind="box", height=6, aspect=2.4)
    plt.xticks(rotation=90)
    plt.title(f"{col} vs crop type")
    save_fig(f"eda_box_{col}", loop=True)
    plt.show()

### 8h. Correlation heat map
Pearson correlation between every pair of features. Strongly correlated features carry overlapping information.

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df[feature_cols].corr(), center=0, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation matrix of the features")
save_fig("05_correlation_heatmap")
plt.show()

## Step 9. Label encoding and train/test split

**What:** turns the crop names into numbers (0 to 21) and splits the data into 80 % training and 20 % test samples.
**Why:** classifiers need numeric targets. The test samples are never used to fit a model, so the test accuracy estimates how well a model handles new data.

**Citation:** `LabelEncoder` and `train_test_split` from scikit-learn (Pedregosa et al., 2011).

In [ ]:
label_encoder = LabelEncoder()
X = df[feature_cols]
y = label_encoder.fit_transform(df["label"])
class_names = label_encoder.classes_
label_dict = {i: name for i, name in enumerate(class_names)}
print(X.shape, y.shape)
print(label_dict)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=TEST_SIZE, random_state=SEED_SPLIT)

print(f"Train data: {X_train.shape}, {y_train.shape}")
print(f"Test data : {X_test.shape}, {y_test.shape}")

## Step 10. Model evaluation helper

**What:** one function that fits a model, scores it and draws its confusion matrices, so that all six classifiers are evaluated in exactly the same way.

**What is reported for each model:**
* **Test accuracy:** share of the held-out test samples classified correctly. **This is the figure that estimates performance on new data.**
* **Weighted F1 (test):** F1 score averaged over crops, weighted by crop size (Pedregosa et al., 2011).
* **Whole-data accuracy:** accuracy on all samples, **including the training samples**. It is reported because the paper reports it, but it is optimistic and should not be read as a measure of generalisation.
* **Confusion matrix:** rows are true crops and columns are predicted crops; off-diagonal cells are mistakes.

Every pipeline first standardises the features (`StandardScaler`: subtract the mean, divide by the standard deviation), fitted on the training data only.

In [ ]:
results = []

def evaluate_model(name, pipeline, figsize=(14, 10)):
    pipeline.fit(X_train, y_train)

    # --- held-out test data ---
    pred_test = pipeline.predict(X_test)
    acc_test = accuracy_score(y_test, pred_test)
    f1_test = f1_score(y_test, pred_test, average="weighted")
    print(f"{name}: accuracy on test data = {acc_test * 100:.2f} %  (weighted F1 = {f1_test:.4f})")
    plt.figure(figsize=figsize)
    sns.heatmap(confusion_matrix(y_test, pred_test), annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{name}: confusion matrix (test data)")
    plt.xlabel("Predicted crop"); plt.ylabel("True crop")
    save_fig(f"cm_test_{name.split()[0].lower()}")
    plt.show()

    # --- all samples (includes the training samples) ---
    pred_all = pipeline.predict(X.values)
    acc_all = accuracy_score(y, pred_all)
    print(f"{name}: accuracy on whole data = {acc_all * 100:.2f} %  (includes training samples)")
    plt.figure(figsize=figsize)
    sns.heatmap(confusion_matrix(y, pred_all), annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{name}: confusion matrix (whole data)")
    plt.xlabel("Predicted crop"); plt.ylabel("True crop")
    plt.show()

    results.append({"model": name,
                    "test_accuracy": acc_test * 100,
                    "test_weighted_f1": f1_test,
                    "whole_data_accuracy": acc_all * 100})
    return pipeline

## Step 11. The six classifiers

### 11a. Logistic Regression
**What:** a linear model that estimates the probability of each crop from a weighted sum of the features (multinomial logistic regression).
**Why:** a simple, interpretable baseline.
**Citation:** Cox (1958); implementation by scikit-learn (Pedregosa et al., 2011).

In [ ]:
lr_pipeline = evaluate_model(
    "Logistic Regression",
    make_pipeline(StandardScaler(), LogisticRegression(random_state=2)))

### 11b. K-Nearest Neighbors (KNN)
**What:** classifies a sample by the majority crop among its `k` closest training samples (distance computed on the standardised features).
**Why:** a non-parametric method that makes no assumption about the shape of the class boundaries.
**Citation:** Cover & Hart (1967); implementation by scikit-learn (Pedregosa et al., 2011).

**Choosing k:** the first cell tries k = 1 to 49 and plots the test error. In the original run the lowest error was at **k = 7** (95.45 %), while the paper uses **k = 3** (95.23 %). The difference is small. The paper's value is kept (`K_FINAL = 3` in Step 2) so that the reported numbers reproduce. Note that choosing k by looking at the test error makes the test score slightly optimistic; Step 13 compares all models without this effect.

In [ ]:
error_rate = []
k_values = list(range(1, 50))
for k in k_values:
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    pipe.fit(X_train, y_train)
    error_rate.append(np.mean(pipe.predict(X_test) != y_test))

best_k = k_values[int(np.argmin(error_rate))]
plt.figure(figsize=(10, 6))
plt.plot(k_values, error_rate, color="blue", linestyle="dashed", marker="o",
         markerfacecolor="red", markersize=6)
plt.title("Test error rate vs. K value")
plt.xlabel("K"); plt.ylabel("Error rate")
save_fig("06_knn_error_vs_k")
plt.show()
print(f"Minimum test error {min(error_rate):.4f} at K = {best_k}")

In [ ]:
knn_pipeline = evaluate_model(
    f"K-Nearest Neighbors (k={K_FINAL})",
    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=K_FINAL)))

### 11c. Random Forest
**What:** an ensemble of many decision trees, each trained on a random bootstrap sample and random subsets of features; the forest votes on the crop.
**Why:** usually accurate and robust, and handles features on different scales without difficulty.
**Citation:** Breiman (2001); implementation by scikit-learn (Pedregosa et al., 2011).

In [ ]:
rf_pipeline = evaluate_model(
    "Random Forest",
    make_pipeline(StandardScaler(), RandomForestClassifier(random_state=18)))

### 11d. XGBoost (gradient boosting)
**What:** builds decision trees one after another, each one correcting the errors of the previous trees.
**Why:** a strong, widely used boosting method for tabular data.
**Citation:** Chen & Guestrin (2016).

In [ ]:
xgb_pipeline = evaluate_model(
    "XGBoost",
    make_pipeline(StandardScaler(), XGBClassifier(random_state=18)))

### 11e. Decision Tree
**What:** a single tree of yes/no questions on the features (for example "is potassium above a threshold?"). Here the splits use the entropy (information gain) criterion and the depth is limited to 5 to avoid over-fitting.
**Why:** easy to read and explain to non-specialists.
**Citation:** Quinlan (1986); implementation by scikit-learn (Pedregosa et al., 2011).

In [ ]:
dt_pipeline = evaluate_model(
    "Decision Tree",
    make_pipeline(StandardScaler(),
                  DecisionTreeClassifier(criterion="entropy", random_state=2, max_depth=5)))

### 11f. Gaussian Naive Bayes
**What:** applies Bayes' rule assuming that, within each crop, every feature follows a normal (Gaussian) distribution and that the features are independent.
**Why:** a fast probabilistic baseline.
**Citation:** Hastie, Tibshirani & Friedman (2009); implementation by scikit-learn (Pedregosa et al., 2011).

In [ ]:
nb_pipeline = evaluate_model(
    "Gaussian Naive Bayes",
    make_pipeline(StandardScaler(), GaussianNB()))

## Step 12. Model comparison

**What:** puts the results of all six models in one table and in two bar charts (test data and whole data), and saves the table to `results/model_comparison.csv`.
**How to read it:** compare models mainly on **test accuracy**. Whole-data accuracy includes training samples. The two bar charts are **Fig. 4** (whole data) and **Fig. 5** (test data) of the paper.

In [ ]:
results_df = pd.DataFrame(results).round(2)
results_df.to_csv(RESULTS / "model_comparison.csv", index=False)
results_df

In [ ]:
for column, title, fname in [("test_accuracy", "Accuracy on test data", "07_accuracy_test"),
                             ("whole_data_accuracy", "Accuracy on whole data (includes training samples)", "08_accuracy_whole")]:
    plt.figure(figsize=(12, 6), dpi=100)
    ax = sns.barplot(data=results_df, x=column, y="model", hue="model", palette="dark", legend=False)
    for i, v in enumerate(results_df[column]):
        ax.text(v, i, f" {v:.2f}", va="center")
    plt.title(title)
    plt.xlabel("Accuracy (%)"); plt.ylabel("Algorithm")
    plt.xlim(0, 110)
    save_fig(fname)
    plt.show()

## Step 13. Robustness check with cross-validation (additional to Figs. 4 and 5)

**What:** repeats the comparison with **stratified 5-fold cross-validation**. The data are split five times; each time four parts train the model and the fifth tests it, so every sample is used for testing exactly once. The mean and standard deviation over the five folds are reported.

**Why this is stricter than the single split:**
* the imputation (mean filling) is done **inside** each training fold, so no information from the test fold leaks in;
* the number of neighbours is fixed in advance (`K_FINAL`), not chosen on the test data;
* the result does not depend on one lucky or unlucky split.

Very high accuracies, such as 100 % for Random Forest and XGBoost on a single split, should be checked this way. If the cross-validated accuracy is also very high, the crops are genuinely well separated in this dataset.

**Citation:** `StratifiedKFold` and `cross_val_score` from scikit-learn (Pedregosa et al., 2011).

In [ ]:
X_raw = df_raw[feature_cols].values          # raw values, gaps still present
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

estimators = {
    "Logistic Regression": LogisticRegression(random_state=2),
    f"K-Nearest Neighbors (k={K_FINAL})": KNeighborsClassifier(n_neighbors=K_FINAL),
    "Random Forest": RandomForestClassifier(random_state=18),
    "XGBoost": XGBClassifier(random_state=18),
    "Decision Tree": DecisionTreeClassifier(criterion="entropy", random_state=2, max_depth=5),
    "Gaussian Naive Bayes": GaussianNB(),
}

cv_rows = []
for name, est in estimators.items():
    pipe = make_pipeline(SimpleImputer(strategy="mean"), StandardScaler(), est)
    scores = cross_val_score(pipe, X_raw, y, cv=cv, scoring="accuracy")
    cv_rows.append({"model": name,
                    "cv_mean_accuracy": scores.mean() * 100,
                    "cv_std": scores.std() * 100})

cv_df = pd.DataFrame(cv_rows).round(2)
cv_df.to_csv(RESULTS / "model_comparison_cv.csv", index=False)
cv_df

## Step 14. Limitations

* The dataset is small (2,200 samples for 22 crops). High accuracy here does not prove the same accuracy on real farms in other regions, seasons or soil types; external validation on independent field data is needed.
* Whole-data accuracy includes training samples and is optimistic; use test or cross-validated accuracy.
* In the single-split analysis, mean imputation was done before the split and k for KNN was inspected on the test set. Step 13 removes both effects.
* The models predict a suitable crop from measurements; they do not model causes. Fertilizer or yield recommendations should be confirmed by agronomists and local soil tests.

## References

* Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5-32.
* Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785-794.
* Cover, T., & Hart, P. (1967). Nearest neighbor pattern classification. *IEEE Transactions on Information Theory*, 13(1), 21-27.
* Cox, D. R. (1958). The regression analysis of binary sequences. *Journal of the Royal Statistical Society: Series B*, 20(2), 215-242.
* Harris, C. R., et al. (2020). Array programming with NumPy. *Nature*, 585, 357-362.
* Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer.
* Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering*, 9(3), 90-95.
* Ingle, A. (n.d.). *Crop Recommendation Dataset.* Kaggle. [verify the exact URL and licence on Kaggle before publishing]
* McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56-61.
* Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825-2830.
* Quinlan, J. R. (1986). Induction of decision trees. *Machine Learning*, 1(1), 81-106.
* Tukey, J. W. (1977). *Exploratory Data Analysis.* Addison-Wesley.
* Waskom, M. L. (2021). seaborn: statistical data visualization. *Journal of Open Source Software*, 6(60), 3021.